# Adapter mit dem Basismodell zusammenführen

GPU-Runtime wählen. Jeder Run wird mit seiner protokollierten Basisrevision geladen, der Adapter
zusammengeführt und als `merged_hf/` neben dem Adapter gespeichert. Danach
[`colab_convert_merged_for_omlx.ipynb`](colab_convert_merged_for_omlx.ipynb) ausführen.

In [ ]:
%pip install -q -U "torchao>=0.16.0" "transformers==5.16.1" "peft==0.20.0" "accelerate==1.15.0" safetensors huggingface_hub

In [ ]:
import gc, json, shutil
from getpass import getpass
from pathlib import Path

import torch
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/llm_phishing_finetune_workflow_v3')
HF_CACHE = Path('/content/hf_merge_runtime_cache/hub')

MERGE_ALL_RUNS = True  # False: nur RUN_RELATIVE_PATH
RUN_RELATIVE_PATH = 'Qwen--Qwen3-4B/phishing_origin/run2_trainall_seq1024_b641f8bce155'
RUN_RELATIVE_PATHS = [
    'Qwen--Qwen3-4B/phishing_origin/run2_trainall_seq1024_b641f8bce155',
    'Qwen--Qwen3-4B/injection/run2_trainall_seq1024_ed25182458b1',
    'Qwen--Qwen3-8B/phishing_origin/run2_trainall_seq1024_e8e2f938c346',
    'Qwen--Qwen3-8B/injection/run2_trainall_seq1024_c95e53553674',
    'microsoft--Phi-4-mini-instruct/phishing_origin/run2_trainall_seq1024_ec6e28568e92',
    'microsoft--Phi-4-mini-instruct/injection/run2_trainall_seq1024_6844caf0fa03',
    'mistralai--Mistral-7B-Instruct-v0.3/phishing_origin/run2_trainall_seq1024_f3f4198653be',
    'mistralai--Mistral-7B-Instruct-v0.3/injection/run2_trainall_seq1024_e61fde90cc14',
    'meta-llama--Llama-3.1-8B-Instruct/phishing_origin/run2_trainall_seq1024_da4398c35e8c',
    'meta-llama--Llama-3.1-8B-Instruct/injection/run2_trainall_seq1024_0cfa9daf5a6c',
]

In [ ]:
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merge_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
hf_token = None

for relative in (RUN_RELATIVE_PATHS if MERGE_ALL_RUNS else [RUN_RELATIVE_PATH]):
    run = PROJECT / 'runs_v4' / relative
    merged_dir = run / 'merged_hf'
    if merged_dir.exists():
        print('Bereits vorhanden:', relative)
        continue
    model_id = json.loads((run / 'run_config.json').read_text())['model_id']
    task = json.loads((run / 'run_config.json').read_text())['task']
    revision = json.loads((run / 'model_revision.json').read_text())['resolved_commit']
    token = False
    if model_id.startswith('meta-llama/'):
        hf_token = hf_token or getpass('Hugging Face Read-Token: ').strip()
        token = hf_token
    snapshot = snapshot_download(repo_id=model_id, revision=revision, cache_dir=str(HF_CACHE), token=token,
                                 allow_patterns=['config.json', 'generation_config.json', '*.safetensors', '*.safetensors.index.json'])
    base = AutoModelForCausalLM.from_pretrained(snapshot, dtype=merge_dtype, device_map='auto',
                                                local_files_only=True, low_cpu_mem_usage=True)
    # Adapter aus dem lokalen Runtime-Speicher statt aus dem Drive-Mount laden.
    local_adapter = Path('/content/merge_adapter')
    shutil.rmtree(local_adapter, ignore_errors=True)
    shutil.copytree(run / 'adapter', local_adapter)
    merged = PeftModel.from_pretrained(base, local_adapter, is_trainable=False).merge_and_unload(safe_merge=True)
    merged.save_pretrained(merged_dir, safe_serialization=True, max_shard_size='2GB')
    AutoTokenizer.from_pretrained(run / 'adapter').save_pretrained(merged_dir)
    (merged_dir / 'artifact_provenance.json').write_text(json.dumps({
        'base_model_id': model_id, 'base_revision': revision, 'task': task, 'merge_dtype': str(merge_dtype),
        'adapter_source': str(run / 'adapter'), 'input_mode': 'text_only',
        'export_model_type': merged.config.model_type}, indent=2))
    del base, merged
    gc.collect()
    torch.cuda.empty_cache()
    print('Zusammengeführt:', merged_dir)